In [2]:
!pip install playwright

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/
     ---------------------------------------- 0.0/38.2 MB ? eta -:--:--
     ----------------- --------------------- 17.6/38.2 MB 82.4 MB/s eta 0:00:01
     ------------------------------- ------- 31.2/38.2 MB 75.1 MB/s eta 0:00:01
     ---------------------------------------- 38.2/38.2 MB 60.3 MB/s  0:00:00

   ------------- -------------------------- 1/3 [greenlet]
   ------------- -------------------------- 1/3 [greenlet]
   -------------------------- ------------- 2/3 [playwright]
   -------------------------- ------------- 2/3 [playwright]
   -------------------------- ------------- 2/3 [playwright]
   -------------------------- ------------- 2/3 [playwright]
   -------------------------- ------------- 2/3 [playwright]
   -------------------------- ------------- 2/3 [playwright]
   -------------------------- ------------- 2/3 [playwri

In [3]:
!playwright install chromium

|                                                                                |   0% of 191.8 MiB
|■■■■■■■■                                                                        |  10% of 191.8 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                |  60% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                        |  70% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                |  80% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■        |  90% of 

In [4]:
from playwright.async_api import async_playwright

async with async_playwright() as p:

    browser = await p.chromium.launch(
        headless=False
    )

    context = await browser.new_context(
        record_har_path="google.har"
    )

    page = await context.new_page()

    await page.goto(
        "https://www.google.com",
        wait_until="networkidle"
    )

    print("Page title:", await page.title())

    # Wait a few seconds to capture additional traffic
    await page.wait_for_timeout(5000)

    # Closing context writes the HAR file
    await context.close()

    await browser.close()

print("HAR capture completed: google.har")

NotImplementedError: 

Yes. This is a different error.

Your first error was because you used the **Sync Playwright API inside Jupyter**. Now you're correctly using the async API, but Windows/Jupyter is failing when Playwright tries to create the browser subprocess:

```text
NotImplementedError
...
asyncio.create_subprocess_exec(...)
```

This usually happens because of the Windows event loop that VS Code Jupyter is using.

The simplest fix is to **run the Playwright code as a normal `.py` file instead of inside the notebook**.

Create a file such as:

```text
capture_har.py
```

with this code:

```python
import asyncio
from playwright.async_api import async_playwright


async def main():

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=False
        )

        context = await browser.new_context(
            record_har_path="google.har"
        )

        page = await context.new_page()

        print("Opening Google...")

        await page.goto(
            "https://www.google.com",
            wait_until="networkidle"
        )

        print("Page title:", await page.title())

        await page.wait_for_timeout(5000)

        await context.close()
        await browser.close()

        print("HAR saved successfully: google.har")


if __name__ == "__main__":
    asyncio.run(main())
```

Then open the VS Code terminal and run:

```powershell
python capture_har.py
```

You should see Chromium open and Google load.

Expected output:

```text
Opening Google...
Page title: Google
HAR saved successfully: google.har
```

Then in your folder you'll have:

```text
capture_har.py
google.har
```

Before running it, make sure Chromium for Playwright is installed:

```powershell
python -m playwright install chromium
```

If that command gives an issue, also check:

```powershell
python -m playwright --version
```

### Why your notebook fails

Your code itself:

```python
async with async_playwright() as p:
```

is valid.

The traceback shows the failure occurs here internally:

```text
asyncio.create_subprocess_exec(...)
```

and then:

```text
raise NotImplementedError
```

Playwright needs to start a browser executable as a child process:

```text
Python
   |
   | create_subprocess_exec()
   v
Playwright
   |
   v
Chromium.exe
   |
   v
https://www.google.com
```

Your current VS Code Jupyter/Windows event-loop environment isn't supporting that subprocess operation correctly.

### You can still analyze the HAR in your notebook

A good setup is:

```text
VS Code Terminal
      |
      | python capture_har.py
      v
   google.har
      |
      v
Jupyter Notebook
      |
      +-- Parse HAR
      +-- Find HTTP errors
      +-- Check latency
      +-- Analyze DNS/TCP/TLS timing
```

For example, after `google.har` is created, your notebook can read it normally:

```python
import json

with open("google.har", "r", encoding="utf-8") as f:
    har = json.load(f)

entries = har["log"]["entries"]

print("Total requests:", len(entries))

for entry in entries:
    request = entry["request"]
    response = entry["response"]

    print(
        response["status"],
        round(entry["time"], 2),
        "ms",
        request["method"],
        request["url"]
    )
```

So I recommend using **`.py` for HAR capture and Jupyter for HAR analysis**. That's also a cleaner workflow for a network automation project.


In [6]:
!python capture_har.py

Opening Google...
Page title: Google
HAR saved successfully: google.har


In [7]:
import json

with open("google.har", "r", encoding="utf-8") as f:
    har = json.load(f)

entries = har["log"]["entries"]

print("Total requests:", len(entries))

for entry in entries:
    request = entry["request"]
    response = entry["response"]

    print(
        response["status"],
        round(entry["time"], 2),
        "ms",
        request["method"],
        request["url"]
    )

Total requests: 39
200 183.32 ms GET https://www.google.com/
200 112.87 ms GET https://www.google.com/xjs/_/ss/k=xjs.hd.I2ysgVDfmfg.L.B1.O/am=AAAEgAAAAAAAAAAAAAgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAQAAAMAQAAAAABgAIAAAAACAAAEQNRAAAAgAAAAAAEA4AAAAYAAAAAAAAAAAgAAAAAABAAACQAAAAAAAAAABAAAAAAAAAAAACAAAAAAEgAAoAACCAAgAAAAAAAIAAAAAAAAAAEAICAAAAAAAAAAAAACAAAAAAAAAAAAACA4AAAAAAwgAAAAAAAAAAAAAAAAAAAAAAAAAAiARIAAAEAAAAAIAAAAAAAAQACCEIAAQogIAFAQAAAAAAFgAAAAAAgAZAgBAgAAAAAAAAAABAACAIhAACAAAAAAEAACAAIAAgAIAAAgCMoAAASFQAAQCAQAAAAAACAAAAAIAAAAAAAAAAAAAAEQAAAAAAAAAAAFgAgAFCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACg/d=1/ed=1/br=1/rs=ACT90oG4wcs0OEGQL_kwL_wluCc3t3ztNA/m=cdos,hsm,jsa,mb4ZUb,cEt90b,SNUn3,qddgKe,sTsDMc,dtl0hd,eHDfl,YV5bee,d,csi?cb=121509378
200 71.72 ms GET https://www.google.com/xjs/_/js/k=xjs.hd.en.8LMLoI0IboA.2019.O/am=AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAQCCAAAAAgCAAAAAAAAAAAAAAAAgABAAAAAAAAAAAAAAAAAAAgABBgACQAAAAAAAAAABAAAAAACAAAAAAAIABAAEgAAoAAICBAAAAAAAA

In [8]:
!python microsoft_har.py

Opening Microsoft...
Page title: Microsoft – AI, Cloud, Productivity, Computing, Gaming & Apps
HAR saved successfully: microsoft.har


In [9]:
import json

with open("microsoft.har", "r", encoding="utf-8") as f:
    har = json.load(f)

entries = har["log"]["entries"]

print("Total requests:", len(entries))

for entry in entries:
    request = entry["request"]
    response = entry["response"]

    print(
        response["status"],
        round(entry["time"], 2),
        "ms",
        request["method"],
        request["url"]
    )

Total requests: 275
302 619.86 ms GET https://www.microsoft.com/
200 113.44 ms GET https://www.microsoft.com/en-us
200 575.72 ms GET https://rum.hlx.page/.rum/@adobe/helix-rum-js@%5E2/dist/micro.js
200 35.91 ms GET https://www.microsoft.com/echo/etc.clientlibs/store/clientlibs/clientlib-reimagine/page/base.ACSHASHf9f0062b1dfdc0b83030b55540e5008f.min.css
200 41.87 ms GET https://www.microsoft.com/echo/etc.clientlibs/store/clientlibs/clientlib-reimagine/themes/store.ACSHASH94d661eddae45b14c9f57e0b79a7fe67.min.css
200 35.28 ms GET https://www.microsoft.com/echo/etc.clientlibs/store/clientlibs/clientlib-reimagine/page/web-components.ACSHASHe4bcd693aea4756a2a970d28ce83902f.min.css
200 37.09 ms GET https://www.microsoft.com/echo/etc.clientlibs/store/clientlibs/clientlib-reimagine-env/base.ACSHASH8cd32d1ad0be8a1a9b98c8f1eda1aae6.min.js
200 35.59 ms GET https://www.microsoft.com/echo/etc.clientlibs/cascade.component.authoring/clientlibs/clientlib-uhf.ACSHASHf9f2395c582fa601707b7a5dfae9f05f.min